# 02 Hidden patterns (local, synthetic + your manual inputs)
Nothing leaves your machine. All profiles synthetic (seed 42). Figures saved to `../benchmarks/`.

In [ ]:
import sys; sys.path.insert(0, "../src")
from credit_truth import fico_simulator as sim
from credit_truth import claims as C
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import json, os
os.makedirs("../benchmarks", exist_ok=True)
print("imports ok")

In [ ]:
# P1+P7: utilization cliffs (aggregate) + per-card penalty
utils=[0.0,0.05,0.15,0.30,0.50,0.75,0.95]
scores=[sim.score(sim.Profile(agg_util=u, max_indiv_util=min(0.95,u+0.05))) for u in utils]
print(list(zip(utils,scores)))
plt.figure(); plt.plot([u*100 for u in utils], scores, marker="o")
plt.xlabel("Aggregate utilization %"); plt.ylabel("Simulator score (educational)")
plt.title("Utilization cliffs: 1-9% optimal, 0% slightly suboptimal")
plt.savefig("../benchmarks/util_cliffs.png", dpi=150, bbox_inches="tight"); print("saved util_cliffs.png")
# per-card vs aggregate: same 20% aggregate, one maxed vs distributed
agg20_distrib=sim.score(sim.Profile(agg_util=0.20, max_indiv_util=0.25))
agg20_maxed=sim.score(sim.Profile(agg_util=0.20, max_indiv_util=0.95))
print(f"agg20 distributed={agg20_distrib} vs one-maxed={agg20_maxed} delta={agg20_maxed-agg20_distrib} (expect -20)")
assert agg20_maxed < agg20_distrib

In [ ]:
# P3: late-payment asymmetry + decay
clean=sim.score(sim.Profile())
for m in [1, 13, 30]:
    s=sim.score(sim.Profile(n30=1, months_since_last_late=m))
    print(f"months_since={m}: score={s} drop={clean-s}")
late_clean_drop=clean-sim.score(sim.Profile(n30=1, months_since_last_late=1))
thin_base=sim.score(sim.Profile(n30=0, collections=1, months_since_last_late=1, agg_util=0.30))
thin_plus=sim.score(sim.Profile(n30=1, collections=1, months_since_last_late=1, agg_util=0.30))
print(f"clean-file 30d hit={late_clean_drop}; dirty-file incremental={(thin_base-thin_plus)}")
# decay check
d1=clean-sim.score(sim.Profile(n30=1, months_since_last_late=1))
d24=clean-sim.score(sim.Profile(n30=1, months_since_last_late=30))
assert d24 < d1, (d1,d24)
print("decay CONFIRMED")
months=[1,6,12,18,24,36]; drops=[clean-sim.score(sim.Profile(n30=1, months_since_last_late=m)) for m in months]
plt.figure(); plt.plot(months, drops, marker="o"); plt.xlabel("Months since 30d late"); plt.ylabel("Points lost vs clean")
plt.title("Late-payment decay (educational)"); plt.savefig("../benchmarks/late_decay.png", dpi=150, bbox_inches="tight"); print("saved late_decay.png")

In [ ]:
# Full claim table re-run (14 claims) + save JSON snapshot for paper
results=[fn() for fn in C.ALL]
for r in results:
    print(f"{r['id']:4s} {r['verdict']:32s} g={r['general_gain']}")
assert results[0]["general_gain"]==0  # C01 letters-alone general = 0
print("C01 REFUTED-as-general holds: letters alone +0 without behavior/error fix")

In [ ]:
# YOUR live check (manual, optional): enter YOUR numbers from AnnualCreditReport.com — nothing uploaded
MY_UTIL=0.29  # <-- replace with your aggregate utilization, e.g. 0.12
MY_MAX=0.35   # <-- replace with your max per-card utilization
mine=sim.score(sim.Profile(agg_util=MY_UTIL, max_indiv_util=MY_MAX))
opt=sim.score(sim.Profile(agg_util=0.05, max_indiv_util=0.08))
print(f"your-sim-proxy={mine} vs optimized-util-proxy={opt} (illustrative gap={opt-mine})")
print("To verify on your reality: pull free weekly reports at AnnualCreditReport.com, compute balances/limits, re-run this cell.")